# SymPy verifications of the main theorems from Section 5

Theorem, algorithm and equation numbers are those of the manuscript version of 24 November 2026. The next cell setups SymPy.

In [1]:
import sympy as sp

def check(expr):
    return sp.simplify(expr)

## 1 Verification of Theorem 5.2

### 1.1 Statement and proof (as in the manuscript)

The statement uses the inequality (3.3), which holds for $f\in\mathcal{F}_{0,L}$ and all $x,y\in\mathbb{R}^d$,
$$
f(y)\geq f(x)+\langle\nabla f(x),\,y-x\rangle+\tfrac{1}{2L}\|\nabla f(y)-\nabla f(x)\|^2 ,
$$
the relation (2.16),
$$
\langle\nabla f(y_k),\,A_kx_k+a_kz_k-A_{k+1}y_k\rangle\geq0 ,
$$
and the potential (5.13),
$$
\Phi_k=A_k\Big(f(y_{k-1})-\tfrac{1}{2L}\|\nabla f(y_{k-1})\|^2-f_\star\Big)+\tfrac L2\|z_k-x_\star\|^2, \quad k\geq1,
$$
with $A_0=0$ so that $\Phi_0=\tfrac{L}{2}\|z_0-x_\star\|^2$.

**Theorem 5.2. (Optimized gradient method, $m=0$)** Let $f\in\mathcal{F}_{0,L}$, $k\geq1$, $A_k,a_k\geq0$, $A_{k+1}=A_k+a_k$, $y_{k-1},z_k,y_k\in\mathbb{R}^d$, $x_k=y_{k-1}-\tfrac1L\nabla f(y_{k-1})$, and let $x_k,z_k,y_k$ satisfy the relation (2.16). Define $x_{k+1}=y_k-\tfrac1L\nabla f(y_k)$, $z_{k+1}=z_k-\tfrac{a_k}{L}\nabla f(y_k)$, and $\Phi_k$ as in (5.13). If $a_k^2\leq2A_{k+1}$, then it holds that:
$$
\Phi_{k+1}\leqslant\Phi_k .
$$

*Proof.* The proof consists of a weighted sum of the following inequalities:

- inequality (3.3) between $y_{k-1}$ and $y_k$, with weight $\lambda_1=A_k$:

  $$
  f(y_{k-1})\geq f(y_k)+\langle\nabla f(y_k),\,y_{k-1}-y_k\rangle+\tfrac{1}{2L}\|\nabla f(y_{k-1})-\nabla f(y_k)\|^2,
  $$

- inequality (3.3) between $x_\star$ and $y_k$, with weight $\lambda_2=a_k$:

  $$
  f_\star\geq f(y_k)+\langle\nabla f(y_k),\,x_\star-y_k\rangle+\tfrac{1}{2L}\|\nabla f(y_k)\|^2,
  $$

- the relation (2.16), with weight $\lambda_3=1$:

  $$
  \langle\nabla f(y_k),\,A_kx_k+a_kz_k-A_{k+1}y_k\rangle\geq0.
  $$

The weighted sum reads

$$
\begin{aligned}
 0\geq{}&\lambda_1\big[f(y_k)-f(y_{k-1})+\langle\nabla f(y_k),\,y_{k-1}-y_k\rangle+\tfrac{1}{2L}\|\nabla f(y_{k-1})-\nabla f(y_k)\|^2\big]\\
 &+\lambda_2\big[f(y_k)-f_\star+\langle\nabla f(y_k),\,x_\star-y_k\rangle+\tfrac{1}{2L}\|\nabla f(y_k)\|^2\big]\\
 &+\lambda_3\langle\nabla f(y_k),\,A_{k+1}y_k-A_kx_k-a_kz_k\rangle,
\end{aligned}
$$

which, using $x_k=y_{k-1}-\tfrac1L\nabla f(y_{k-1})$, $z_{k+1}=z_k-\tfrac{a_k}{L}\nabla f(y_k)$ and $A_{k+1}=A_k+a_k$, can be reformulated exactly as

$$
 0\geq\Phi_{k+1}-\Phi_k+\tfrac{2A_{k+1}-a_k^2}{2L}\|\nabla f(y_k)\|^2 ,
$$

which concludes the proof by using $a_k^2\leq2A_{k+1}$. $\square$

### 1.2 SymPy verification

To verify the expression symbolically, one can define the following symbols:

In [2]:
L = sp.symbols('L')
Ak = sp.symbols('A_k')
ak = sp.symbols('a_k')

zk = sp.symbols('z_k')
yk = sp.symbols('y_k')
ykm1 = sp.symbols('y_{k-1}')
xs = sp.symbols('x_\\star')
gyk = sp.symbols('\\nabla{}f(y_k)')
gykm1 = sp.symbols('\\nabla{}f(y_{k-1})')
fyk = sp.symbols('f(y_k)')
fykm1 = sp.symbols('f(y_{k-1})')
fs = sp.symbols('f_\\star')

Akp1 = Ak + ak
xk = ykm1 - 1 / L * gykm1
zkp1 = zk - ak / L * gyk

Now, one can write the three inequalities used in the proof (in the form "$\ldots\leqslant 0$") and their weights to form the weighted sum:

In [3]:
inequality1 = fyk - fykm1 + gyk * (ykm1 - yk) + 1 / 2 / L * (gykm1 - gyk) ** 2
inequality2 = fyk - fs + gyk * (xs - yk) + 1 / 2 / L * gyk ** 2
inequality3 = - gyk * (Ak * xk + ak * zk - Akp1 * yk)
lambda1 = Ak
lambda2 = ak
lambda3 = 1

weighted_sum = lambda1 * inequality1 + lambda2 * inequality2 + lambda3 * inequality3

Similarly, one can define the two potential values and form the target inequality

In [4]:
potentialk = Ak * (fykm1 - 1 / 2 / L * gykm1 ** 2 - fs) + L / 2 * (zk - xs) ** 2
potentialkp1 = Akp1 * (fyk - 1 / 2 / L * gyk ** 2 - fs) + L / 2 * (zkp1 - xs) ** 2

target_inequality = potentialkp1 - potentialk + (2 * Akp1 - ak ** 2) / 2 / L * gyk ** 2

It remains to check that the expressions match, which we check by verifying their difference is identically zero

In [5]:
check(weighted_sum - target_inequality)

0

As complements, on Algorithm 5.1, on the first iteration and on the last step. In the order of the checks below:

- in Algorithm 5.1, $y_k=(1-\tau_k)\,x_k+\tau_k\,z_k$ with $\tau_k=a_k/(A_k+a_k)$, so that $A_kx_k+a_kz_k-A_{k+1}y_k=0$: the relation (2.16) holds with equality, whatever the increment, hence also for the output (5.11);
- the increment of (5.10), $a_k=1+\sqrt{1+2A_k}$, satisfies the condition of the theorem with equality, $a_k^2=2A_{k+1}$;
- compared with Nesterov's FGM (2.18), the increments and the weights are doubled: if $a=\tfrac{1+\sqrt{4A+1}}{2}$ is the increment of (2.18) at the weight $A$, then the increment of (5.10) at the weight $2A$ is $2a$, and the coefficient $\tau_k$ is the same in both methods;
- the first iteration ($k=0$, $A_0=0$ and $y_0=z_0=x_0$): the same computation without the first inequality gives $0\geq\Phi_1-\Phi_0+\tfrac{2a_0-a_0^2}{2L}\|\nabla f(y_0)\|^2$, hence $\Phi_1\leq\Phi_0$ as soon as $a_0^2\leq2a_0$;
- the last step: the weighted sum of the proof with $k=N$ can also be reformulated as (5.14),
$$
 0\geq A_{N+1}\big(f(y_N)-f_\star\big)+\tfrac L2\|z_{N+1}-x_\star\|^2-\Phi_N+\tfrac{A_{N+1}-a_N^2}{2L}\|\nabla f(y_N)\|^2 ,
$$
with $z_{N+1}=z_N-\tfrac{a_N}{L}\nabla f(y_N)$ and $A_{N+1}=A_N+a_N$ for a last increment $a_N\geq0$;
- the output (5.11) uses the largest increment such that $a_N^2\leq A_N+a_N$, that is, $\tilde a_N=\tfrac{1+\sqrt{1+4A_N}}{2}$;
- for Algorithm 5.7 (OGM-G), whose sequence is $\theta_k=a_k/2$ for $k<N$ and $\theta_N=\tilde a_N$: as $A_k=a_{k-1}^2/2=2\theta_{k-1}^2$, the increments give $\theta_k=\tfrac{1+\sqrt{1+4\theta_{k-1}^2}}{2}$ and $\theta_N=\tfrac{1+\sqrt{1+8\theta_{N-1}^2}}{2}$, with $\theta_0=a_0/2=1$ and $A_N+\tilde a_N=\tilde a_N^2=\theta_N^2$;
- for Corollary 5.3: $A_k+1+\sqrt{2A_k}=\big(\sqrt{A_k}+\tfrac{1}{\sqrt2}\big)^2+\tfrac12$, so that $a_k\geq1+\sqrt{2A_k}$ gives $A_{k+1}\geq\big(\sqrt{A_k}+\tfrac{1}{\sqrt2}\big)^2$, and the first weight is $A_1=a_0=2$ (last output below).

In [6]:
A, theta = sp.symbols('A theta', positive=True)
tauk = ak / (Ak + ak)                       # coefficient of Algorithm 5.1
ak_ogm = 1 + sp.sqrt(1 + 2 * Ak)            # increment of (5.10)
ak_fgm = (1 + sp.sqrt(4 * A + 1)) / 2       # increment of Nesterov's FGM (2.18) at the weight A
aN_output = (1 + sp.sqrt(1 + 4 * Ak)) / 2   # increment of the output (5.11), A_k in the role of A_N

potential_last = Akp1 * (fyk - fs) + L / 2 * (zkp1 - xs) ** 2          # first two terms of (5.14)
target_last_step = potential_last - potentialk + (Akp1 - ak ** 2) / 2 / L * gyk ** 2

(check((Ak * xk + ak * zk - Akp1 * yk).subs(yk, (1 - tauk) * xk + tauk * zk)),
 check(ak_ogm ** 2 - 2 * (Ak + ak_ogm)),
 check(ak_ogm.subs(Ak, 2 * A) - 2 * ak_fgm),
 check(tauk.subs(ak, ak_ogm).subs(Ak, 2 * A) - ak_fgm / (A + ak_fgm)),
 check((lambda2 * inequality2 - target_inequality).subs({Ak: 0, yk: zk})),
 check(weighted_sum - target_last_step),
 check(aN_output ** 2 - (Ak + aN_output)),
 check(ak_ogm.subs(Ak, 2 * theta ** 2) / 2 - (1 + sp.sqrt(1 + 4 * theta ** 2)) / 2),
 check(aN_output.subs(Ak, 2 * theta ** 2) - (1 + sp.sqrt(1 + 8 * theta ** 2)) / 2),
 check(A + 1 + sp.sqrt(2 * A) - (sp.sqrt(A) + 1 / sp.sqrt(2)) ** 2 - sp.Rational(1, 2)),
 ak_ogm.subs(Ak, 0))

(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2)

As complements, on the model-based interpretation (Section 5.3.2). The pieces (5.15) are $q_i(x)=f(y_i)+\langle\nabla f(y_i),\,x-y_i\rangle+\tfrac{1}{2L}\|\nabla f(y_i)\|^2$, the quantity tracked on the upper side is $u_k(x_k)=f(y_{k-1})-\tfrac{1}{2L}\|\nabla f(y_{k-1})\|^2$, and the regularized lower model is
$$
A_k\,r_k(x)=\tfrac L2\|x-x_0\|^2+\sum_{i=0}^{k-1}a_i\,q_i(x) .
$$
It is a quadratic with curvature $L$. Below, it is written with its minimizer $z_k$ and with the gap (5.17), $\Psi_k=A_k\big(u_k(x_k)-r_k(z_k)\big)$, which is a free symbol: $A_k\,r_k(x)=A_k\,u_k(x_k)-\Psi_k+\tfrac L2\|x-z_k\|^2$. In the order of the checks:

- the second inequality of the proof is $q_k(x_\star)-f_\star\leq0$;
- induction on the minimizer: if $z_k$ minimizes $r_k$, then $z_{k+1}=z_k-\tfrac{a_k}{L}\nabla f(y_k)$ minimizes $A_{k+1}\,r_{k+1}=A_k\,r_k+a_k\,q_k$;
- the weighted sum of the first and third inequalities of the proof, with the same weights, can be reformulated exactly as $0\geq\Psi_{k+1}-\Psi_k+\tfrac{2A_{k+1}-a_k^2}{2L}\|\nabla f(y_k)\|^2$;
- the first iteration ($A_0=0$, $\Psi_0=0$ and $y_0=z_0$, no inequality is used): $\Psi_1=-\tfrac{2a_0-a_0^2}{2L}\|\nabla f(y_0)\|^2$, which is nonpositive as soon as $a_0^2\leq2a_0$, so that $\Psi_1\leq0$;
- the relation with the potential, $\Phi_k=\Psi_k+\tfrac L2\|x_0-x_\star\|^2-\sum_{i=0}^{k-1}a_i\big(f_\star-q_i(x_\star)\big)$, checked on its increments; as $q_i(x_\star)\leq f_\star$, it gives $\Phi_k\leq\Psi_k+\tfrac L2\|x_0-x_\star\|^2$, which is (5.16).

In [7]:
x = sp.symbols('x')
Psik = sp.symbols('Psi_k')

def q_piece(y, g, f, x):
    return f + g * (x - y) + g ** 2 / 2 / L        # piece (5.15) of the query (y, g, f)

uk = fykm1 - gykm1 ** 2 / 2 / L                    # u_k(x_k)
ukp1 = fyk - gyk ** 2 / 2 / L                      # u_{k+1}(x_{k+1})
rk_x = Ak * uk - Psik + L / 2 * (x - zk) ** 2      # A_k r_k(x), minimized at z_k
rkp1_x = rk_x + ak * q_piece(yk, gyk, fyk, x)      # A_{k+1} r_{k+1}(x)
Psikp1 = Akp1 * ukp1 - rkp1_x.subs(x, zkp1)
leftover_ogm = (2 * Akp1 - ak ** 2) / 2 / L * gyk ** 2

(check(inequality2 - (q_piece(yk, gyk, fyk, xs) - fs)),
 check(sp.diff(rkp1_x, x).subs(x, zkp1)),
 check(lambda1 * inequality1 + lambda3 * inequality3 - (Psikp1 - Psik + leftover_ogm)),
 check(Psikp1.subs({Ak: 0, Psik: 0, yk: zk}) + (2 * ak - ak ** 2) / 2 / L * gyk ** 2),
 check((potentialkp1 - potentialk) - (Psikp1 - Psik) + ak * (fs - q_piece(yk, gyk, fyk, xs))))

(0, 0, 0, 0, 0)

## 2 Verification of Theorem 5.5

### 2.1 Statement and proof (as in the manuscript)

The statement uses the improved descent lemma (5.22), with $x^+=x-\tfrac1L\nabla f(x)$,
$$
f(x^+)\leq f(x)-\tfrac{1}{2L}\|\nabla f(x)\|^2-\tfrac{1}{2L}\|\nabla f(x^+)\|^2 ,
$$
the monotonicity (5.23) of the gradient norms along gradient descent with step size $1/L$,
$$
\|\nabla f(x_{k+1})\|\leq\|\nabla f(x_k)\| ,
$$
and the potential (5.24),
$$
\Phi_k=A_k\big(f(x_k)-f_\star\big)+\tfrac{B_k}{2L}\|\nabla f(x_k)\|^2+\tfrac L2\|x_k-x_\star\|^2 .
$$

**Theorem 5.5. (Gradient descent, gradient norms)** Let $f\in\mathcal{F}_{0,L}$, let $k\geq0$, $x_k\in\mathbb{R}^d$, $x_{k+1}=x_k-\tfrac1L\nabla f(x_k)$, $A_k,B_k\geq 0$, and let $\Phi_k$ be given by (5.24). It holds that:
$$
\Phi_{k+1}\leqslant\Phi_k,
$$
with $A_{k+1}=A_k+1$ and $B_{k+1}=B_k+A_k+A_{k+1}$.

*Proof.* The proof consists of a weighted sum of the following inequalities:

- convexity (2.1) between $x_\star$ and $x_k$, with weight $\lambda_1=A_{k+1}-A_k=1$:

  $$
  f_\star\geq f(x_k)+\langle\nabla f(x_k),\,x_\star-x_k\rangle,
  $$

- the improved descent lemma (5.22) at $x_k$, with weight $\lambda_2=A_{k+1}$:

  $$
  f(x_{k+1})\leq f(x_k)-\tfrac{1}{2L}\|\nabla f(x_k)\|^2-\tfrac{1}{2L}\|\nabla f(x_{k+1})\|^2,
  $$

- monotonicity of the gradient norms (5.23), with weight $\lambda_3=\tfrac{A_k+B_k}{2L}$:

  $$
  \|\nabla f(x_{k+1})\|^2\leq\|\nabla f(x_k)\|^2 .
  $$

The first two inequalities are from the proof of Theorem 2.2, with the descent lemma replaced by its improved version. The weighted sum of the three inequalities reads

$$
\begin{aligned}
0\geq{}& \lambda_1\big[f(x_k)-f_\star+\langle\nabla f(x_k),\,x_\star-x_k\rangle\big]\\
& + \lambda_2\big[f(x_{k+1})- f(x_k)+\tfrac{1}{2L}\|\nabla f(x_k)\|^2+\tfrac{1}{2L}\|\nabla f(x_{k+1})\|^2\big]\\
& + \lambda_3 \big[\|\nabla f(x_{k+1})\|^2-\|\nabla f(x_k)\|^2\big],
\end{aligned}
$$

which, using the update $x_{k+1}=x_k-\tfrac1L\nabla f(x_k)$ and $B_{k+1}=B_k+A_k+A_{k+1}$, can be reformulated exactly as

$$
 0\geq\Phi_{k+1}-\Phi_k,
$$

by the same steps as in the proof of Theorem 2.2, concluding the proof. $\square$

### 2.2 SymPy verification

To verify the expression symbolically, one can define the following symbols:

In [8]:
L = sp.symbols('L')
Ak = sp.symbols('A_k')
Bk = sp.symbols('B_k')

xk = sp.symbols('x_k')
xs = sp.symbols('x_\\star')
gk = sp.symbols('\\nabla{}f(x_k)')
gkp1 = sp.symbols('\\nabla{}f(x_{k+1})')
fk = sp.symbols('f(x_k)')
fkp1 = sp.symbols('f(x_{k+1})')
fs = sp.symbols('f_\\star')

Akp1 = Ak + 1
Bkp1 = Bk + Ak + Akp1
xkp1 = xk - 1 / L * gk

Now, as before, one can write the inequalities used in the proof (in the form "$\ldots\leqslant 0$") and their weights to form the weighted sum:

In [9]:
inequality1 = fk - fs + gk * (xs - xk)
inequality2 = fkp1 - fk + 1 / 2 / L * gk ** 2 + 1 / 2 / L * gkp1 ** 2
inequality3 = gkp1 ** 2 - gk ** 2
lambda1 = 1
lambda2 = Akp1
lambda3 = (Ak + Bk) / 2 / L

weighted_sum = lambda1 * inequality1 + lambda2 * inequality2 + lambda3 * inequality3

Similarly, one can define the two potential values and form the target inequality

In [10]:
potentialk = Ak * (fk - fs) + Bk / 2 / L * gk ** 2 + L / 2 * (xk - xs) ** 2
potentialkp1 = Akp1 * (fkp1 - fs) + Bkp1 / 2 / L * gkp1 ** 2 + L / 2 * (xkp1 - xs) ** 2

target_inequality = potentialkp1 - potentialk

It remains to check that the expressions match, which we check by verifying their difference is identically zero

In [11]:
check(weighted_sum - target_inequality)

0

As complements, in the order of the checks below:

- the improved descent lemma (5.22) is the inequality (3.3) between $x$ and $x^+=x-\tfrac1L\nabla f(x)$: the two expressions are the same;
- replacing the descent lemma (2.3) by its improved version (5.22) in the weighted sum of Theorem 2.2 (weights $1$ and $A_{k+1}$) leaves one more term in the reformulation, which becomes $0\geq\Phi_{k+1}-\Phi_k+\tfrac{A_k}{2L}\|\nabla f(x_k)\|^2+\tfrac{A_{k+1}}{2L}\|\nabla f(x_{k+1})\|^2$ for the potential (2.5), $\Phi_k=A_k\big(f(x_k)-f_\star\big)+\tfrac L2\|x_k-x_\star\|^2$;
- monotonicity of the gradient norms (5.23): the sum of (3.3) between $x_k$ and $x_{k+1}$ and between $x_{k+1}$ and $x_k$ is $\tfrac1L\big(\|\nabla f(x_{k+1})\|^2-\langle\nabla f(x_{k+1}),\,\nabla f(x_k)\rangle\big)\leq0$, hence (5.23) by Cauchy–Schwarz;
- for Corollary 5.6: $A_k=k$ and $B_k=k^2$ satisfy the two recursions of the theorem;
- accelerated methods: replacing the descent lemma by its improved version (5.22) in the weighted sum of Theorem 2.5 (weights $a_k$, $A_k$, $A_{k+1}$ and $1$, potential (2.15), $\Phi_k=A_k\big(f(x_k)-f_\star\big)+\tfrac L2\|z_k-x_\star\|^2$) only changes one thing, the reformulation gets the additional term $\tfrac{A_{k+1}}{2L}\|\nabla f(x_{k+1})\|^2$:
$$
 0\geq\Phi_{k+1}+\tfrac{A_{k+1}}{2L}\|\nabla f(x_{k+1})\|^2-\Phi_k+\tfrac{A_{k+1}-a_k^2}{2L}\|\nabla f(y_k)\|^2 ;
$$
- $\sum_{k=1}^{N}\tfrac{k^2}{4}-\tfrac{N^3}{12}=\tfrac{N(3N+1)}{24}\geq0$ (last output below), so that $A_k\geq k^2/4$ gives $\sum_{k=1}^NA_k\geq N^3/12$.

In [12]:
x, gx, gxp, fx, fxp = sp.symbols('x \\nabla{}f(x) \\nabla{}f(x^+) f(x) f(x^+)')
xp = x - gx / L                                                           # x^+
interpolation = fxp - fx + gxp * (x - xp) + 1 / 2 / L * (gx - gxp) ** 2   # (3.3) between x and x^+
improved_descent = fxp - fx + 1 / 2 / L * gx ** 2 + 1 / 2 / L * gxp ** 2  # (5.22)

potentialk_gd = Ak * (fk - fs) + L / 2 * (xk - xs) ** 2                   # potential (2.5)
potentialkp1_gd = Akp1 * (fkp1 - fs) + L / 2 * (xkp1 - xs) ** 2
target_gd = potentialkp1_gd - potentialk_gd + Ak / 2 / L * gk ** 2 + Akp1 / 2 / L * gkp1 ** 2

# sum of (3.3) between x_k and x_{k+1} and between x_{k+1} and x_k
sum_of_two = ((fkp1 - fk + gkp1 * (xk - xkp1) + 1 / 2 / L * (gk - gkp1) ** 2)
              + (fk - fkp1 + gk * (xkp1 - xk) + 1 / 2 / L * (gk - gkp1) ** 2))

kk, N = sp.symbols('k N')

ak = sp.symbols('a_k')
zk = sp.symbols('z_k')
yk = sp.symbols('y_k')
gyk = sp.symbols('\\nabla{}f(y_k)')
fyk = sp.symbols('f(y_k)')
fxk = sp.symbols('f(x_k)')
Akp1_cg = Ak + ak
zkp1 = zk - ak / L * gyk
inequality1_cg = fyk - fs + gyk * (xs - yk)                                  # convexity between x_* and y_k
inequality2_cg = fyk - fxk + gyk * (xk - yk)                                 # convexity between x_k and y_k
inequality3_cg = fkp1 - fyk + 1 / 2 / L * gyk ** 2 + 1 / 2 / L * gkp1 ** 2   # improved descent lemma at y_k
inequality4_cg = - gyk * (Ak * xk + ak * zk - Akp1_cg * yk)                  # the relation (2.16)
weighted_sum_cg = ak * inequality1_cg + Ak * inequality2_cg + Akp1_cg * inequality3_cg + inequality4_cg
potentialk_cg = Ak * (fxk - fs) + L / 2 * (zk - xs) ** 2                     # potential (2.15)
potentialkp1_cg = Akp1_cg * (fkp1 - fs) + L / 2 * (zkp1 - xs) ** 2
leftover_cg = (Akp1_cg - ak ** 2) / 2 / L * gyk ** 2
target_cg = potentialkp1_cg + Akp1_cg / 2 / L * gkp1 ** 2 - potentialk_cg + leftover_cg

(check(interpolation - improved_descent),
 check(lambda1 * inequality1 + lambda2 * inequality2 - target_gd),
 check(sum_of_two - 1 / L * (gkp1 ** 2 - gkp1 * gk)),
 check(Akp1.subs(Ak, kk) - (kk + 1)),
 check(Bkp1.subs({Ak: kk, Bk: kk ** 2}) - (kk + 1) ** 2),
 check(weighted_sum_cg - target_cg),
 sp.factor(sp.summation(kk ** 2 / 4, (kk, 1, N)) - N ** 3 / 12))

(0, 0, 0, 0, 0, 0, N*(3*N + 1)/24)

## 3 Verification of Theorem 5.9

### 3.1 Statement and proof (as in the manuscript)

The statement uses the inequality (3.4), which holds for $f\in\mathcal{F}_{m,L}$ and all $x,y\in\mathbb{R}^d$,
$$
f(y)\geq f(x)+\langle\nabla f(x),\,y-x\rangle+\tfrac{1}{2L}\|\nabla f(y)-\nabla f(x)\|^2+\tfrac{mL}{2(L-m)}\big\|y-x-\tfrac1L\big(\nabla f(y)-\nabla f(x)\big)\big\|^2 ,
$$
where $\tfrac{mL}{2(L-m)}=\tfrac{m}{2(1-q)}$ with $q=m/L$, the following method, whose two displays are (5.28) and (5.29), and the potential (5.31).

**Algorithm 5.8. (Information-theoretic exact method (ITEM))** Set $x_0=z_0$ and $A_0=0$:
$$
\left\{\,
\begin{aligned}
 y_k &= (1-\tau_k) \, x_k+ \tau_k \, z_k,\\
 x_{k+1}&=y_k-\tfrac1L\nabla f(y_k),\\
 z_{k+1}&=z_k+q\delta_k\,(y_k-z_k)-\tfrac{\delta_k}{L}\nabla f(y_k),
\end{aligned}
\right.
\ \ \text{with}\ \
\begin{aligned}
 \tau_k &= 1-\tfrac{A_k}{(1-q)A_{k+1}},\\
 \delta_k&=\tfrac{a_k+qA_{k+1}}{1+2qA_{k+1}},
\end{aligned}
$$
where $A_{k+1}=A_k+a_k$, and $a_k$ is the largest increment such that
$$
(a_k+qA_{k+1})^2\leq2A_{k+1}(1+2qA_{k+1}),
$$
namely
$$
 a_k=\tfrac{1+q(3-q)A_k+\sqrt{(1+2A_k)(1+2qA_k)}}{(1-q)^2}.
$$

Recalling that $x_k=y_{k-1}-\tfrac1L\nabla f(y_{k-1})$, the potential (5.31) is
$$
\begin{aligned}
 \Phi_k={}&A_k\Big[(1-q)\Big(f(y_{k-1})-\tfrac{1}{2L}\|\nabla f(y_{k-1})\|^2-f_\star\Big)-\tfrac m2\|x_k-x_\star\|^2\Big]\\
 &+\tfrac{L+2mA_k}{2}\|z_k-x_\star\|^2, \qquad k\geq1,
\end{aligned}
$$
with $A_0=0$ so that $\Phi_0=\tfrac{L}{2}\|z_0-x_\star\|^2$.

**Theorem 5.9. (ITEM, $m\geq0$)** Let $f\in\mathcal{F}_{m,L}$, $k\geq1$, $A_k>0$, $a_k\geq0$, $A_{k+1}=A_k+a_k$, $y_{k-1},z_k\in\mathbb{R}^d$, and $x_k=y_{k-1}-\tfrac1L\nabla f(y_{k-1})$. Define $y_k$, $x_{k+1}$ and $z_{k+1}$ as in (5.28), and $\Phi_k$ as in (5.31). If $(a_k+qA_{k+1})^2\leq2A_{k+1}(1+2qA_{k+1})$, then it holds that:
$$
\Phi_{k+1}\leqslant\Phi_k .
$$

*Proof.* The proof consists of a weighted sum of the following inequalities, in which the last term of (3.4) is written with the gradient steps $x_k$ and $x_{k+1}$:

- inequality (3.4) between $y_{k-1}$ and $y_k$, with weight $\lambda_1=(1-q)A_k$:

  $$
  f(y_{k-1})\geq f(y_k)+\langle\nabla f(y_k),\,y_{k-1}-y_k\rangle+\tfrac{1}{2L}\|\nabla f(y_{k-1})-\nabla f(y_k)\|^2+\tfrac{m}{2(1-q)}\|x_k-x_{k+1}\|^2,
  $$

- inequality (3.4) between $x_\star$ and $y_k$, with weight $\lambda_2=(1-q)a_k$:

  $$
  f_\star\geq f(y_k)+\langle\nabla f(y_k),\,x_\star-y_k\rangle+\tfrac{1}{2L}\|\nabla f(y_k)\|^2+\tfrac{m}{2(1-q)}\|x_\star-x_{k+1}\|^2.
  $$

The weighted sum reads

$$
\begin{aligned}
 0\geq{}&\lambda_1\big[f(y_k)-f(y_{k-1})+\langle\nabla f(y_k),\,y_{k-1}-y_k\rangle+\tfrac{1}{2L}\|\nabla f(y_{k-1})-\nabla f(y_k)\|^2+\tfrac{m}{2(1-q)}\|x_k-x_{k+1}\|^2\big]\\
 &+\lambda_2\big[f(y_k)-f_\star+\langle\nabla f(y_k),\,x_\star-y_k\rangle+\tfrac{1}{2L}\|\nabla f(y_k)\|^2+\tfrac{m}{2(1-q)}\|x_\star-x_{k+1}\|^2\big],
\end{aligned}
$$

which, using the updates (5.28) and $A_{k+1}=A_k+a_k$, can be reformulated exactly as

$$
 0\geq\Phi_{k+1}-\Phi_k+\tfrac{2A_{k+1}-\delta_k(a_k+qA_{k+1})}{2L}\|\nabla f(y_k)-m\,(y_k-z_k)\|^2 ,
$$

which is the claim, since $\delta_k(a_k+qA_{k+1})\leq2A_{k+1}$ is our assumption on the increments. $\square$

### 3.2 SymPy verification

To verify the expression symbolically, one can define the following symbols:

In [13]:
L = sp.symbols('L')
m = sp.symbols('m')
Ak = sp.symbols('A_k')
ak = sp.symbols('a_k')

zk = sp.symbols('z_k')
ykm1 = sp.symbols('y_{k-1}')
xs = sp.symbols('x_\\star')
gyk = sp.symbols('\\nabla{}f(y_k)')
gykm1 = sp.symbols('\\nabla{}f(y_{k-1})')
fyk = sp.symbols('f(y_k)')
fykm1 = sp.symbols('f(y_{k-1})')
fs = sp.symbols('f_\\star')

q = m / L
Akp1 = Ak + ak
tauk = 1 - Ak / (1 - q) / Akp1
deltak = (ak + q * Akp1) / (1 + 2 * q * Akp1)

xk = ykm1 - gykm1 / L
yk = (1 - tauk) * xk + tauk * zk
xkp1 = yk - gyk / L
zkp1 = zk + q * deltak * (yk - zk) - deltak / L * gyk

Now, one can write the two inequalities used in the proof (in the form "$\ldots\leqslant 0$") and their weights to form the weighted sum:

In [14]:
inequality1 = fyk - fykm1 + gyk * (ykm1 - yk) + (gykm1 - gyk) ** 2 / 2 / L + m / 2 / (1 - q) * (xk - xkp1) ** 2
inequality2 = fyk - fs + gyk * (xs - yk) + gyk ** 2 / 2 / L + m / 2 / (1 - q) * (xs - xkp1) ** 2
lambda1 = (1 - q) * Ak
lambda2 = (1 - q) * ak

weighted_sum = lambda1 * inequality1 + lambda2 * inequality2

Similarly, one can define the two potential values and form the target inequality

In [15]:
potentialk = Ak * ((1 - q) * (fykm1 - gykm1 ** 2 / 2 / L - fs) - m / 2 * (xk - xs) ** 2) + (L + 2 * m * Ak) / 2 * (zk - xs) ** 2
potentialkp1 = Akp1 * ((1 - q) * (fyk - gyk ** 2 / 2 / L - fs) - m / 2 * (xkp1 - xs) ** 2) + (L + 2 * m * Akp1) / 2 * (zkp1 - xs) ** 2

leftover = (2 * Akp1 - deltak * (ak + q * Akp1)) / 2 / L * (gyk - m * (yk - zk)) ** 2
target_inequality = potentialkp1 - potentialk + leftover

It remains to check that the expressions match, which we check by verifying their difference is identically zero

In [16]:
check(weighted_sum - target_inequality)

0

As complements, on Algorithm 5.8, on the first iteration, and on Corollary 5.10 with the bound (5.30),
$$
 \|z_N-x_\star\|^2\leq\frac{\|x_0-x_\star\|^2}{1+2qA_N}\leq\frac{(1-\sqrt q)^{2N}}{(1-\sqrt q)^{2N}+q}\,\|x_0-x_\star\|^2 .
$$
In the order of the checks below:

- the coefficient of the leftover term is $\tfrac{2A_{k+1}-\delta_k(a_k+qA_{k+1})}{2L}$, with $2A_{k+1}-\delta_k(a_k+qA_{k+1})=\tfrac{2A_{k+1}(1+2qA_{k+1})-(a_k+qA_{k+1})^2}{1+2qA_{k+1}}$, which is nonnegative exactly under the condition of the theorem;
- the increment (5.29) satisfies this condition with equality, and corresponds to $A_{k+1}=\tfrac{(1+q)A_k+1+\sqrt{(1+2A_k)(1+2qA_k)}}{(1-q)^2}$;
- for $m=0$, the coefficients are those of OGM (5.10): $a_k=1+\sqrt{1+2A_k}$, $\tau_k=a_k/A_{k+1}$ and $\delta_k=a_k$;
- the first iteration ($k=0$, $A_0=0$): then $\tau_0=1$, that is, $y_0=z_0$, the increment (5.29) is $a_0=2/(1-q)^2$, and the identity checked above (in which the first inequality has weight zero) leaves the coefficient $\tfrac{a_0\,(2-(1-q)^2a_0)}{2L(1+2qa_0)}$, which is nonnegative as soon as $a_0\leq2/(1-q)^2$;
- growth of the weights, for Corollary 5.10: $(a_k+qA_{k+1})^2-4qA_{k+1}^2$ is $2A_{k+1}\geq0$ when the condition holds with equality, so that $a_k+qA_{k+1}\geq2\sqrt q\,A_{k+1}$, that is, $A_k\leq(1-\sqrt q)^2A_{k+1}$; and the first weight satisfies $2A_1(1-\sqrt q)^2=\tfrac{4}{(1+\sqrt q)^2}\geq1$, hence $2A_N\geq(1-\sqrt q)^{-2N}$;
- the bound (5.30): chaining gives $\tfrac{L+2mA_N}{2}\|z_N-x_\star\|^2\leq\tfrac L2\|x_0-x_\star\|^2$, which is the first inequality as $\tfrac{L}{L+2mA_N}=\tfrac{1}{1+2qA_N}$; and for $2A_N=1/\rho$, with $\rho$ in the role of $(1-\sqrt q)^{2N}$, $\tfrac{1}{1+2qA_N}=\tfrac{\rho}{\rho+q}$, which is the second one, as the left-hand side decreases with $A_N$.

In [17]:
condition = 2 * Akp1 * (1 + 2 * q * Akp1) - (ak + q * Akp1) ** 2         # must be nonnegative
ak_item = (1 + q * (3 - q) * Ak + sp.sqrt((1 + 2 * Ak) * (1 + 2 * q * Ak))) / (1 - q) ** 2   # increment (5.29)
Akp1_item = ((1 + q) * Ak + 1 + sp.sqrt((1 + 2 * Ak) * (1 + 2 * q * Ak))) / (1 - q) ** 2
sq = sp.sqrt(q)
AN, rho = sp.symbols('A_N rho')

(check((2 * Akp1 - deltak * (ak + q * Akp1)) - condition / (1 + 2 * q * Akp1)),
 check(condition.subs(ak, ak_item)),
 check(Ak + ak_item - Akp1_item),
 check(ak_item.subs(m, 0) - (1 + sp.sqrt(1 + 2 * Ak))),
 check(tauk.subs(m, 0) - ak / Akp1),
 check(deltak.subs(m, 0) - ak),
 check(tauk.subs(Ak, 0) - 1),
 check(ak_item.subs(Ak, 0) - 2 / (1 - q) ** 2),
 check(((2 * Akp1 - deltak * (ak + q * Akp1)) / 2 / L).subs(Ak, 0) - ak * (2 - (1 - q) ** 2 * ak) / 2 / L / (1 + 2 * q * ak)),
 check((ak + q * Akp1) ** 2 - 4 * q * Akp1 ** 2 - (2 * Akp1 - condition)),
 check(Akp1 - (2 * sq - q) * Akp1 - (1 - sq) ** 2 * Akp1),
 check(2 * ak_item.subs(Ak, 0) * (1 - sq) ** 2 - 4 / (1 + sq) ** 2),
 check(L / (L + 2 * m * AN) - 1 / (1 + 2 * q * AN)),
 check((1 / (1 + 2 * q * AN)).subs(AN, 1 / (2 * rho)) - rho / (rho + q)))

(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0)

### 3.3 Relation with Lemma 1 of Taylor and Drori (2023)

Theorem 5.9 is Lemma 1 of Taylor and Drori (2023, [arXiv:2101.09741](https://arxiv.org/abs/2101.09741)), written in the format and with the normalization of the previous sections, and the two statements do not coincide term by term. We write $\hat A_k$ and $\hat\delta_k$ for the weights and the coefficient of that paper, which uses $\mu$ for $m$. Its Algorithm 1 starts from $\hat A_0=0$ and iterates
$$
y_k=(1-\beta_k)\,z_k+\beta_k\,x_k,\qquad x_{k+1}=y_k-\tfrac1L\nabla f(y_k),\qquad z_{k+1}=(1-q\hat\delta_k)\,z_k+q\hat\delta_k\,y_k-\tfrac{\hat\delta_k}{L}\nabla f(y_k),
$$
with
$$
\hat A_{k+1}=\frac{(1+q)\hat A_k+2\big(1+\sqrt{(1+\hat A_k)(1+q\hat A_k)}\big)}{(1-q)^2},\qquad \beta_k=\frac{\hat A_k}{(1-q)\hat A_{k+1}},\qquad \hat\delta_k=\frac12\,\frac{(1-q)^2\hat A_{k+1}-(1+q)\hat A_k}{1+q+q\hat A_k},
$$
and its Lemma 1 states that $\phi_{k+1}\leq\phi_k$ for these parameters and all $k\geq0$, with
$$
\begin{aligned}
\phi_k&=(1-q)\hat A_k\,\psi_{k-1}+(L+m\hat A_k)\|z_k-x_\star\|^2,\\
\psi_{k-1}&=f(y_{k-1})-f_\star-\tfrac{1}{2L}\|\nabla f(y_{k-1})\|^2-\tfrac{m}{2(1-q)}\big\|y_{k-1}-\tfrac1L\nabla f(y_{k-1})-x_\star\big\|^2 .
\end{aligned}
$$
The differences with Algorithm 5.8 and Theorem 5.9 are the following.

- **Normalization.** The weights and the potential of the paper are twice ours: $\hat A_k=2A_k$ and $\phi_k=2\Phi_k$. The manuscript keeps the normalization of its other potentials, $A_k\,(\ldots)+\tfrac L2\|z_k-x_\star\|^2$, so that for $m=0$ the method (5.28)–(5.29) is OGM (5.10) and the potential (5.31) is (5.13). The recursion on $\hat A_{k+1}$ is the increment (5.29), and the bound $\|z_N-x_\star\|^2\leq\|z_0-x_\star\|^2/(1+q\hat A_N)$ of the paper is the first bound of (5.30).
- **Names.** The iterates are the same: $\beta_k=1-\tau_k$, and Lemma 1 writes $y_{k-1}-\tfrac1L\nabla f(y_{k-1})$ for the point $x_k$. Its proof uses the same two inequalities, in the other order, with the weights $(1-q)(\hat A_{k+1}-\hat A_k)$ and $(1-q)\hat A_k$, which are twice ours.
- **One weight, or a condition on the increments.** Lemma 1 is about the weight $\hat A_{k+1}$ of the method only, which is the largest root of $P(\hat A_{k+1},\hat A_k)=0$, where $P(x,y)=(y-(1-q)x)^2-4x(1+qy)$. Its proof reformulates the weighted sum, for any $\hat A_{k+1}$, as
$$
\begin{aligned}
\phi_{k+1}\leq{}&\phi_k-LK_1\,P(\hat A_{k+1},\hat A_k)\,\|z_k-x_\star\|^2\\
&+\tfrac{1}{4L}K_2\,P(\hat A_{k+1},\hat A_k)\,\big\|(1-q)\hat A_{k+1}\nabla f(y_k)-m\hat A_k(x_k-x_\star)+K_3\,m\,(z_k-x_\star)\big\|^2 ,
\end{aligned}
$$
with
$$
K_1=\frac{q^2}{(1+q)^2+(1-q)^2q\hat A_{k+1}},\qquad K_2=\frac{(1+q)^2+(1-q)^2q\hat A_{k+1}}{(1-q)^2(1+q+q\hat A_k)^2\hat A_{k+1}^2},\qquad K_3=(1+q)\,\frac{(1+q)\hat A_k-(1-q)(2+q\hat A_k)\hat A_{k+1}}{(1+q)^2+(1-q)^2q\hat A_{k+1}},
$$
and the two last terms vanish at the root. Theorem 5.9 is instead stated as the other one-iteration theorems of the manuscript: for all the increments that satisfy a condition, with a leftover term. This condition is $P(\hat A_{k+1},\hat A_k)\leq0$, as $P(2A_{k+1},2A_k)=-4\big(2A_{k+1}(1+2qA_{k+1})-(a_k+qA_{k+1})^2\big)$. As $K_1$ and $K_2$ are nonnegative, for a weight below the largest one the two last terms of the display above have opposite signs, the first one being nonnegative, and the display does not give $\phi_{k+1}\leq\phi_k$ when $m>0$.
- **The coefficient $\delta_k$.** This is why $\delta_k$ is not written as in the paper. In our weights, $\hat\delta_k=\tfrac{(1-q)^2A_{k+1}-(1+q)A_k}{1+q+2qA_k}$, whereas (5.28) uses $\delta_k=\tfrac{a_k+qA_{k+1}}{1+2qA_{k+1}}$, and
$$
\hat\delta_k-\delta_k=-\frac{2q\,\big(2A_{k+1}(1+2qA_{k+1})-(a_k+qA_{k+1})^2\big)}{(1+q+2qA_k)(1+2qA_{k+1})} .
$$
The two coefficients, and hence the two points $z_{k+1}$, coincide for the increment (5.29), where the condition holds with equality, and when $m=0$. They differ for smaller increments, and with $\delta_k$ the weighted sum leaves a single term, a square whose coefficient is nonnegative under the condition. With the increment (5.29), no term is left in either formulation, and the two identities are the same up to the factor two.
- **First iteration.** Lemma 1 covers $k=0$, with $\hat A_0=0$. Theorem 5.9 is stated for $k\geq1$, and the first iteration is treated after its proof, as for OGM.

The checks are, in this order: the recursion on $\hat A_{k+1}$ at $\hat A_k=2A_k$ gives twice the weight $A_{k+1}$ of (5.29); $\beta_k=1-\tau_k$; $\phi_k=2\Phi_k$, and $\phi_{k+1}=2\Phi_{k+1}$ at the same point $z_{k+1}$; the two inequalities of the proof of Lemma 1 are those of the proof of Theorem 5.9, with twice their weights; $P(2A_{k+1},2A_k)$ is $-4$ times the quantity that the condition requires to be nonnegative, and it vanishes for the increment (5.29); the difference $\hat\delta_k-\delta_k$ displayed above, and the corresponding difference between the two points $z_{k+1}$; the reformulation of the proof of Lemma 1, for any increment; and, for $m=0$, $K_1=0$ and $\hat\delta_k=\delta_k$.

In [18]:
# quantities of Taylor and Drori (2023): their weights are twice ours
Ak_hat, Akp1_hat = 2 * Ak, 2 * Akp1
Ah = sp.symbols('\\hat{}A_k')
Akp1_hat_item = ((1 + q) * Ah + 2 * (1 + sp.sqrt((1 + Ah) * (1 + q * Ah)))) / (1 - q) ** 2    # weight of their Algorithm 1
beta = Ak_hat / (1 - q) / Akp1_hat
delta_hat = ((1 - q) ** 2 * Akp1_hat - (1 + q) * Ak_hat) / (1 + q + q * Ak_hat) / 2
zkp1_hat = (1 - q * delta_hat) * zk + q * delta_hat * yk - delta_hat / L * gyk

def phi_hat(A_hat, y, g, f, z):
    psi = f - fs - g ** 2 / 2 / L - m / 2 / (1 - q) * (y - g / L - xs) ** 2
    return (1 - q) * A_hat * psi + (L + m * A_hat) * (z - xs) ** 2

# the two inequalities of the proof of Lemma 1 (in the form "<= 0") and their weights
inequality1_hat = fyk - fs + gyk * (xs - yk) + gyk ** 2 / 2 / L + m / 2 / (1 - q) * (yk - xs - gyk / L) ** 2
inequality2_hat = (fyk - fykm1 + gyk * (ykm1 - yk) + (gyk - gykm1) ** 2 / 2 / L
                   + m / 2 / (1 - q) * (yk - ykm1 - (gyk - gykm1) / L) ** 2)
lambda1_hat = (1 - q) * (Akp1_hat - Ak_hat)
lambda2_hat = (1 - q) * Ak_hat

# reformulation of the weighted sum in the proof of Lemma 1
P = (Ak_hat - (1 - q) * Akp1_hat) ** 2 - 4 * Akp1_hat * (1 + q * Ak_hat)
K1 = q ** 2 / ((1 + q) ** 2 + (1 - q) ** 2 * q * Akp1_hat)
K2 = ((1 + q) ** 2 + (1 - q) ** 2 * q * Akp1_hat) / (1 - q) ** 2 / (1 + q + q * Ak_hat) ** 2 / Akp1_hat ** 2
K3 = (1 + q) * ((1 + q) * Ak_hat - (1 - q) * (2 + q * Ak_hat) * Akp1_hat) / ((1 + q) ** 2 + (1 - q) ** 2 * q * Akp1_hat)
vector = (1 - q) * Akp1_hat * gyk - m * Ak_hat * (xk - xs) + K3 * m * (zk - xs)
target_hat = (phi_hat(Akp1_hat, yk, gyk, fyk, zkp1_hat) - phi_hat(Ak_hat, ykm1, gykm1, fykm1, zk)
              + L * K1 * P * (zk - xs) ** 2 - K2 * P / 4 / L * vector ** 2)

(check(Akp1_hat_item.subs(Ah, 2 * Ak) - 2 * Akp1_item),
 check(beta - (1 - tauk)),
 check(phi_hat(Ak_hat, ykm1, gykm1, fykm1, zk) - 2 * potentialk),
 check(phi_hat(Akp1_hat, yk, gyk, fyk, zkp1) - 2 * potentialkp1),
 check(inequality1_hat - inequality2),
 check(inequality2_hat - inequality1),
 check(lambda1_hat - 2 * lambda2),
 check(lambda2_hat - 2 * lambda1),
 check(P + 4 * condition),
 check(P.subs(ak, ak_item)),
 check(delta_hat - deltak + 2 * q * condition / (1 + q + 2 * q * Ak) / (1 + 2 * q * Akp1)),
 check(zkp1_hat - zkp1 - (delta_hat - deltak) * (q * (yk - zk) - gyk / L)),
 check(lambda1_hat * inequality1_hat + lambda2_hat * inequality2_hat - target_hat),
 check(K1.subs(m, 0)),
 check((delta_hat - deltak).subs(m, 0)))

(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0)

## 4 Verification of the model-based interpretation in Section 5.6

After Corollary 5.10, Section 5.6 reads ITEM through the template of Section 4, then drops the regularization, which gives a triple momentum method, and finally selects the weights from the observed information, which gives Algorithm 5.11. This part verifies the identities behind these three paragraphs. Each subsection recalls the displays of the manuscript, and then lists the checks in the order of the code cell that follows. The code cells use the symbols, the two inequalities and the potentials of part 3.

### 4.1 Model-based interpretation: (5.32)–(5.35)

The inequality (3.4) between $x_\star$ and $y_i$, and that between $y_i$ and $x_\star$, respectively give $C_i(x_\star)\leq f_\star\leq D_i(x_\star)$, with the pieces (5.32),
$$
\begin{aligned}
 C_i(x)&=f(y_i)+\langle\nabla f(y_i),\,x-y_i\rangle+\tfrac{1}{2L}\|\nabla f(y_i)\|^2+\tfrac{m}{2(1-q)}\|x-x_{i+1}\|^2,\\
 D_i(x)&=f(y_i)-\tfrac{1}{2L}\|\nabla f(y_i)\|^2-\tfrac{m}{2(1-q)}\|x-x_{i+1}\|^2,
\end{aligned}
$$
which are the functions $C_i$ and $D_i$ of (3.16). On the upper side, the quantity that is tracked is $u_k(x_k)=f(y_{k-1})-\tfrac{1}{2L}\|\nabla f(y_{k-1})\|^2$, as for OGM, and $D_{k-1}(x_\star)=u_k(x_k)-\tfrac{m}{2(1-q)}\|x_\star-x_k\|^2$. The last term is moved to the lower side, in the regularized lower model (5.33),
$$
 (1-q)A_k\,r_k(x)=\tfrac L2\|x-x_0\|^2+(1-q)\sum_{i=0}^{k-1}a_i\,C_i(x)+\tfrac{mA_k}{2}\|x-x_k\|^2 ,
$$
which is a quadratic with curvature $L+2mA_k$, whose minimizer is precisely the point $z_k$ of (5.28). As $C_i(x_\star)\leq f_\star\leq D_{k-1}(x_\star)$, this gives (5.34),
$$
\begin{aligned}
 (1-q)A_k\,r_k(z_k)+\tfrac{L+2mA_k}{2}\|z_k-x_\star\|^2&=(1-q)A_k\,r_k(x_\star)\\
 &\leq(1-q)A_k\,u_k(x_k)+\tfrac L2\|x_0-x_\star\|^2 ,
\end{aligned}
$$
and the analysis tracks the gap (5.35),
$$
 \Psi_k=(1-q)A_k\big(u_k(x_k)-r_k(z_k)\big),\qquad \Psi_0=0 .
$$
The first inequality of the proof of Theorem 5.9, with the same weight, can be reformulated as
$$
 0\geq\Psi_{k+1}-\Psi_k+\tfrac{2A_{k+1}-\delta_k(a_k+qA_{k+1})}{2L}\|\nabla f(y_k)-m\,(y_k-z_k)\|^2 ,
$$
which shows that $\Psi_{k+1}\leq\Psi_k$ under the same condition on the increments. We thus reach $\Psi_N\leq0$, and (5.34) gives back the bound $\tfrac{L+2mA_N}{2}\|z_N-x_\star\|^2\leq\tfrac L2\|x_0-x_\star\|^2$ of (5.30).

In the code, the model $r_k$ is written with its minimizer $z_k$ and with the gap $\Psi_k$, which is a free symbol: $(1-q)A_k\,r_k(x)=(1-q)A_k\,u_k(x_k)-\Psi_k+\tfrac{L+2mA_k}{2}\|x-z_k\|^2$. The checks are, in this order:

- the pieces (5.32) are the functions $C_i$ and $D_i$ of Section 3.6, see (3.16), written for the query $(y_k,\nabla f(y_k),f(y_k))$;
- $C_k$ is a quadratic with curvature $\tfrac{m}{1-q}$, minimized at $y_k-\tfrac1m\nabla f(y_k)$, with minimal value $f(y_k)-\tfrac1{2m}\|\nabla f(y_k)\|^2$, as the lower bound $\ell_k$ of (4.1);
- the upper bound is $D_{k-1}(x)=u_k(x_k)-\tfrac{m}{2(1-q)}\|x-x_k\|^2$;
- the second inequality of the proof of Theorem 5.9 is $C_k(x_\star)-f_\star\leq0$, and the term in brackets in the potential (5.31) is $(1-q)\big(D_{k-1}(x_\star)-f_\star\big)$, which is nonnegative;
- induction on the minimizer: if $z_k$ minimizes $r_k$, then the point $z_{k+1}$ of (5.28) is a stationary point of $(1-q)A_{k+1}\,r_{k+1}=(1-q)A_k\,r_k-\tfrac{mA_k}{2}\|\cdot-x_k\|^2+(1-q)\,a_k\,C_k+\tfrac{mA_{k+1}}{2}\|\cdot-x_{k+1}\|^2$, whose curvature is $L+2mA_{k+1}$;
- the first inequality of the proof, with the same weight, can be reformulated exactly as the inequality on the gaps displayed above;
- the first iteration ($A_0=0$ and $\Psi_0=0$, no inequality is used): $\Psi_1=-\tfrac{a_0\,(2-(1-q)^2a_0)}{2L(1+2qa_0)}\|\nabla f(y_0)\|^2$, which is nonpositive as soon as $a_0\leq2/(1-q)^2$;
- the bound (5.34): its right-hand side minus $(1-q)A_k\,r_k(x_\star)$ is $(1-q)\sum_{i=0}^{k-1}a_i\big(f_\star-C_i(x_\star)\big)+(1-q)A_k\big(D_{k-1}(x_\star)-f_\star\big)$, a sum of nonnegative terms, checked on its increments;
- the relation with the potential, $\Phi_k=\Psi_k+\tfrac L2\|x_0-x_\star\|^2-(1-q)\sum_{i=0}^{k-1}a_i\big(f_\star-C_i(x_\star)\big)$, checked on its increments.

In [19]:
x = sp.symbols('x')
Psik = sp.symbols('Psi_k')

def C(y, g, f, x):
    return f + g * (x - y) + g ** 2 / 2 / L + m / 2 / (1 - q) * (x - (y - g / L)) ** 2

def D(y, g, f, x):
    return f - g ** 2 / 2 / L - m / 2 / (1 - q) * (x - (y - g / L)) ** 2

C_section3 = fyk + gyk * (x - yk) + m / 2 * (x - yk) ** 2 + (gyk + m * (x - yk)) ** 2 / 2 / (L - m)
D_section3 = fyk - m / 2 * (x - yk) ** 2 - (gyk + m * (x - yk)) ** 2 / 2 / (L - m)

uk = fykm1 - gykm1 ** 2 / 2 / L                                                  # u_k(x_k)
ukp1 = fyk - gyk ** 2 / 2 / L                                                    # u_{k+1}(x_{k+1})
rk_x = (1 - q) * Ak * uk - Psik + (L + 2 * m * Ak) / 2 * (x - zk) ** 2           # (1-q) A_k r_k(x), minimized at z_k
rkp1_x = rk_x - m * Ak / 2 * (x - xk) ** 2 + (1 - q) * ak * C(yk, gyk, fyk, x) + m * Akp1 / 2 * (x - xkp1) ** 2
Psikp1 = (1 - q) * Akp1 * ukp1 - rkp1_x.subs(x, zkp1)
boundk = (1 - q) * Ak * uk - rk_x.subs(x, xs)                                    # left-hand side of the bound, up to L/2 |x_0 - x_star|^2
boundkp1 = (1 - q) * Akp1 * ukp1 - rkp1_x.subs(x, xs)

(check(C(yk, gyk, fyk, x) - C_section3),
 check(D(yk, gyk, fyk, x) - D_section3),
 check(sp.diff(C(yk, gyk, fyk, x), x, 2) - m / (1 - q)),
 check(sp.diff(C(yk, gyk, fyk, x), x).subs(x, yk - gyk / m)),
 check(C(yk, gyk, fyk, x).subs(x, yk - gyk / m) - (fyk - gyk ** 2 / 2 / m)),
 check(D(ykm1, gykm1, fykm1, x) - (uk - m / 2 / (1 - q) * (x - xk) ** 2)),
 check(inequality2 - (C(yk, gyk, fyk, xs) - fs)),
 check(potentialk - (Ak * (1 - q) * (D(ykm1, gykm1, fykm1, xs) - fs) + (L + 2 * m * Ak) / 2 * (zk - xs) ** 2)),
 check(sp.diff(rkp1_x, x).subs(x, zkp1)),
 check(sp.diff(rkp1_x, x, 2) - (L + 2 * m * Akp1)),
 check(lambda1 * inequality1 - (Psikp1 - Psik + leftover)),
 check(Psikp1.subs({Ak: 0, Psik: 0}) + leftover.subs(Ak, 0)),
 check((boundkp1 - boundk) - (1 - q) * (ak * (fs - C(yk, gyk, fyk, xs)) + Akp1 * (D(yk, gyk, fyk, xs) - fs) - Ak * (D(ykm1, gykm1, fykm1, xs) - fs))),
 check((potentialkp1 - potentialk) - (Psikp1 - Psik) + (1 - q) * ak * (fs - C(yk, gyk, fyk, xs))))

(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0)

### 4.2 Without regularization, a triple momentum method: (5.36)–(5.38)

When $m>0$, the regularization can be dropped. Without the first term of (5.33) and the factor $1-q$, the lower model is itself a quadratic, with curvature $\tfrac{2m}{1-q}$, and $z_k$ and $v_k$ denote its minimizer and its minimal value: this is (5.36),
$$
 l_k(x)=\frac{1}{A_k}\sum_{i=0}^{k-1}a_i\,C_i(x)+\tfrac{m}{2(1-q)}\|x-x_k\|^2=v_k+\tfrac{m}{1-q}\|x-z_k\|^2 .
$$
As $l_k(x_\star)\leq u_k(x_k)$, the gap $\Psi_k=A_k\big(u_k(x_k)-v_k\big)$ is a computable certificate, and the minimizer belongs to the ball (5.37),
$$
 \|z_k-x_\star\|^2\leq\tfrac{1-q}{m}\big(u_k(x_k)-v_k\big).
$$
Now, let the new piece $C_k$ receive the weight $a_k/A_{k+1}=1-(1-\sqrt q)^2$, that is, $A_{k+1}=A_k/(1-\sqrt q)^2$, with $y_k$ as in (5.28). The first inequality of the proof of Theorem 5.9, with weight $A_k$, can then be reformulated exactly as $0\geq\Psi_{k+1}-\Psi_k$, and the method reads (5.38),
$$
\left\{\begin{aligned}
 y_k&=x_k+\tfrac{2\sqrt q}{1+\sqrt q}\,(z_k-x_k),\\
 x_{k+1}&=y_k-\tfrac1L\nabla f(y_k),\\
 z_{k+1}&=(1-\sqrt q)\,z_k+\sqrt q\,\big(y_k-\tfrac1m\nabla f(y_k)\big),
\end{aligned}\right.
$$
which is the triple momentum method. The squared radius of the ball (5.37) shrinks by a factor $(1-\sqrt q)^2$ at each iteration. Starting from a single query at $y_0$, that is, from $x_1=y_0-\tfrac1L\nabla f(y_0)$ and $l_1=C_0+\tfrac{m}{2(1-q)}\|\cdot-x_1\|^2$, we have $z_1=y_0-\tfrac12\big(\tfrac1m+\tfrac1L\big)\nabla f(y_0)$, the first ball is (3.17a), and we obtain, for all $N\geq1$,
$$
 \|z_N-x_\star\|^2\leq(1-\sqrt q)^{2(N-1)}\,\tfrac14\big(\tfrac1m-\tfrac1L\big)^2\|\nabla f(y_0)\|^2 .
$$

In the code, $z_k$ and $v_k$ are free symbols, and the increment $a_k\geq0$ is first left free, with $y_k=(1-\tau_k)\,x_k+\tau_k\,z_k$ and the coefficient $\tau_k$ of (5.28). The checks are, in this order:

- adding a piece, $A_{k+1}\,l_{k+1}=A_k\big(l_k-\tfrac{m}{2(1-q)}\|\cdot-x_k\|^2\big)+a_k\,C_k+\tfrac{mA_{k+1}}{2(1-q)}\|\cdot-x_{k+1}\|^2$, gives a quadratic with curvature $\tfrac{2m}{1-q}$ whose minimizer is $z_{k+1}=(1-\theta_k)\,z_k+\theta_k\big(y_k-\tfrac1m\nabla f(y_k)\big)$ with $\theta_k=\tfrac{a_k+qA_{k+1}}{2A_{k+1}}$;
- the first inequality of the proof, with weight $A_k$, can be reformulated exactly as
$$
0\geq\Psi_{k+1}-\Psi_k+\tfrac{4qA_{k+1}^2-(a_k+qA_{k+1})^2}{4m(1-q)A_{k+1}}\|\nabla f(y_k)-m\,(y_k-z_k)\|^2 ,
$$
so that $\Psi_{k+1}\leq\Psi_k$ as soon as $a_k\leq\big(1-(1-\sqrt q)^2\big)A_{k+1}$;
- for the weight $a_k/A_{k+1}=1-(1-\sqrt q)^2$, that is, $A_{k+1}=A_k/(1-\sqrt q)^2$, the leftover term vanishes and the method is (5.38): $y_k=x_k+\tfrac{2\sqrt q}{1+\sqrt q}(z_k-x_k)$ and $z_{k+1}=(1-\sqrt q)\,z_k+\sqrt q\,\big(y_k-\tfrac1m\nabla f(y_k)\big)$; the inequality $\Psi_{k+1}\leq\Psi_k$ then reads $u_{k+1}(x_{k+1})-v_{k+1}\leq(1-\sqrt q)^2\big(u_k(x_k)-v_k\big)$, which is the factor on the squared radius;
- the first ball: from $x_1=y_0-\tfrac1L\nabla f(y_0)$ and $l_1=C_0+\tfrac{m}{2(1-q)}\|\cdot-x_1\|^2$, the minimizer is $z_1=y_0-\tfrac12\big(\tfrac1m+\tfrac1L\big)\nabla f(y_0)$ and the squared radius of (5.37) is $\tfrac{1-q}{m}\big(u_1(x_1)-v_1\big)=\tfrac14\big(\tfrac1m-\tfrac1L\big)^2\|\nabla f(y_0)\|^2$, as in (3.17a).

In [20]:
vk = sp.symbols('v_k')

lk_x = vk + m / (1 - q) * (x - zk) ** 2                              # lower model l_k
lkp1_x = (Ak * (lk_x - m / 2 / (1 - q) * (x - xk) ** 2) + ak * C(yk, gyk, fyk, x)) / Akp1 + m / 2 / (1 - q) * (x - xkp1) ** 2
thetak = (ak + q * Akp1) / 2 / Akp1
zkp1_tmm = (1 - thetak) * zk + thetak * (yk - gyk / m)
vkp1 = lkp1_x.subs(x, zkp1_tmm)
gapk = Ak * (uk - vk)
gapkp1 = Akp1 * (ukp1 - vkp1)
leftover_tmm = (4 * q * Akp1 ** 2 - (ak + q * Akp1) ** 2) / 4 / m / (1 - q) / Akp1 * (gyk - m * (yk - zk)) ** 2

sq = sp.sqrt(q)
largest = {ak: (1 - (1 - sq) ** 2) * Ak / (1 - sq) ** 2}             # A_{k+1} = A_k / (1 - sqrt q)^2

y0, gy0, fy0 = sp.symbols('y_0 \\nabla{}f(y_0) f(y_0)')
x1 = y0 - gy0 / L
l1_x = C(y0, gy0, fy0, x) + m / 2 / (1 - q) * (x - x1) ** 2
z1 = y0 - (1 / m + 1 / L) / 2 * gy0

(check(sp.diff(lkp1_x, x, 2) - 2 * m / (1 - q)),
 check(sp.diff(lkp1_x, x).subs(x, zkp1_tmm)),
 check(Ak * inequality1 - (gapkp1 - gapk + leftover_tmm)),
 check(Akp1.subs(largest) - Ak / (1 - sq) ** 2),
 check(leftover_tmm.subs(largest)),
 check((yk - (xk + 2 * sq / (1 + sq) * (zk - xk))).subs(largest)),
 check((zkp1_tmm - ((1 - sq) * zk + sq * (yk - gyk / m))).subs(largest)),
 check(sp.diff(l1_x, x).subs(x, z1)),
 check((1 - q) / m * (fy0 - gy0 ** 2 / 2 / L - l1_x.subs(x, z1)) - (1 / m - 1 / L) ** 2 * gy0 ** 2 / 4))

(0, 0, 0, 0, 0, 0, 0, 0, 0)

### 4.3 Adjusting the weights: (5.39)

The weights of the pieces can be selected from the observed information rather than fixed in advance, while keeping a single aggregate: a convex combination $w_k$ of $C_0,\ldots,C_{k-1}$, selected at the previous iteration. With $w_\lambda=(1-\lambda)\,w_k+\lambda\,C_k$, the lower model (5.36) becomes $l_\lambda=w_\lambda+\tfrac{m}{2(1-q)}\|\cdot-x_{k+1}\|^2$, and $l_\lambda(x_\star)\leq u_{k+1}(x_{k+1})$ for all $\lambda\in[0,1]$. With the parameter $\gamma\in[0,1]$ of (4.23) in front of the regularization, the model $l_\lambda+\tfrac{\gamma L}{2(1-q)A}\|\cdot-z_0\|^2$ has the minimizer $z_0-\tfrac tL\,\bar g_\lambda$ and the minimal value $l_\lambda(z_0)-\tfrac{t}{2L}\|\bar g_\lambda\|^2$, as in (4.24), where $\bar g_\lambda=\nabla l_\lambda(z_0)$ and now $t=\tfrac{(1-q)A}{\gamma+2qA}$. Measuring the values relative to the upper bound, we write $e_\lambda=l_\lambda(z_0)-u_{k+1}(x_{k+1})=(1-\lambda)\,e_0+\lambda\,e_1$ and $\bar g_\lambda=(1-\lambda)\,\bar g_0+\lambda\,\bar g_1$, with (5.39),
$$
 \begin{aligned}
 e_0&=w_k(z_0)-D_k(z_0), &\bar g_0&=\nabla w_k(z_0)-\nabla D_k(z_0),\\
 e_1&=C_k(z_0)-D_k(z_0), &\bar g_1&=\nabla f(y_k)+\tfrac{2m}{1-q}\,(z_0-x_{k+1}),
 \end{aligned}
$$
so that this minimal value is $u_{k+1}(x_{k+1})+e_\lambda-\tfrac{t}{2L}\|\bar g_\lambda\|^2$. When $\gamma=1$, this is ITEM with a single aggregate, for $m\geq0$: the certificate $\Psi_{k+1}\leq0$ again reads $e_\lambda-\tfrac{t}{2L}\|\bar g_\lambda\|^2\geq0$, and the largest certified weight follows from the ratio (4.25) and its stationary points (4.26), as in Algorithm 4.14, with $\tilde A_{k+1}=\tfrac{t_{k+1}}{1-q-2q\,t_{k+1}}$ and $z_{k+1}=z_0-\tfrac{t_{k+1}}{L}\,\bar g_{\lambda_k}$. When $\gamma=0$ and $m>0$, we have $t=\tfrac{1-q}{2q}$ for all $A$, the minimal value no longer depends on the regularization, and maximizing it over $\lambda$ gives the weight (4.27) with $\tfrac{2m}{1-q}$ in place of $m$.

In the code, the aggregate $w_k$ is any quadratic with curvature $\tfrac{m}{1-q}$, as all convex combinations of the pieces $C_i$ are, known through the free symbols $w_k(z_0)$ and $\nabla w_k(z_0)$, and the query point is the point $y_k$ of ITEM. The checks are, in this order:

- $l_\lambda(x)-u_{k+1}(x_{k+1})=w_\lambda(x)-D_k(x)=e_\lambda+\langle\bar g_\lambda,\,x-z_0\rangle+\tfrac{m}{1-q}\|x-z_0\|^2$, with $e_\lambda$ and $\bar g_\lambda$ as above, where $\bar g_1=\nabla C_k(z_0)-\nabla D_k(z_0)$;
- the model $l_\lambda+\tfrac{\gamma L}{2(1-q)A}\|\cdot-z_0\|^2$ is stationary at $z_0-\tfrac tL\,\bar g_\lambda$ with $t=\tfrac{(1-q)A}{\gamma+2qA}$, and its minimal value is $u_{k+1}(x_{k+1})+e_\lambda-\tfrac{t}{2L}\|\bar g_\lambda\|^2$; conversely, $A=\tfrac{\gamma\,t}{1-q-2qt}$;
- when $\gamma=1$, at the first iteration ($y_0=z_0$, a single piece), $t_1=2L\,e_1/\|\bar g_1\|^2$ is the a priori value, that of $A_1=2/(1-q)^2$;
- when $\gamma=0$, $t=\tfrac{1-q}{2q}$, the minimizer is $z_0-\tfrac{1-q}{2m}\,\bar g_\lambda$, and the value $e_\lambda-\tfrac{t}{2L}\|\bar g_\lambda\|^2$ is a concave quadratic function of $\lambda$ whose derivative vanishes at the weight of (5.40) before projection, that is, (4.27) with $\tfrac{2m}{1-q}$ in place of $m$;
- still when $\gamma=0$, the squared radius $\tfrac{1-q}{m}\big(u_{k+1}(x_{k+1})-\min l_\lambda\big)$ of the ball (5.37) is $\big(\tfrac{1-q}{2m}\big)^2\|\bar g_\lambda\|^2-\tfrac{1-q}{m}\,e_\lambda$, which is $-\tfrac{1-q}{m}$ times the value above: the weight that maximizes the value gives the smallest ball.

When $\gamma=1$, the a priori weight $\lambda=a_k/A_{k+1}$ is always certified: this is the inequality on the gaps checked in subsection 4.1 of this notebook, which only uses that $r_k$ is a quadratic with curvature $L+2mA_k$, minimizer $z_k$ and $\Psi_k\leq0$.

In [21]:
lam, gam, z0, A, wkz0, gwkz0 = sp.symbols('lambda gamma z_0 A w_k(z_0) \\nabla{}w_k(z_0)')

def ball(y, wz0, gwz0):
    # e_lambda, g_lambda of (5.39) for a query at y and an aggregate given by its value and gradient at z_0,
    # then the center and the squared radius of the ball obtained without regularization
    Ck_x, Dk_x = C(y, gyk, fyk, x), D(y, gyk, fyk, x)
    e0, g0 = wz0 - Dk_x.subs(x, z0), gwz0 - sp.diff(Dk_x, x).subs(x, z0)
    e1, g1 = (Ck_x - Dk_x).subs(x, z0), gyk + 2 * m / (1 - q) * (z0 - (y - gyk / L))
    e, g = (1 - lam) * e0 + lam * e1, (1 - lam) * g0 + lam * g1
    return e0, e1, g0, g1, e, g, z0 - (1 - q) / 2 / m * g, ((1 - q) / 2 / m * g) ** 2 - (1 - q) / m * e

# for any aggregate and any query point (here y_k of ITEM)
e0, e1, g0, g1, elam, glam, zlam, R2lam = ball(yk, wkz0, gwkz0)
wk_x = wkz0 + gwkz0 * (x - z0) + m / 2 / (1 - q) * (x - z0) ** 2
llam_x = (1 - lam) * wk_x + lam * C(yk, gyk, fyk, x) + m / 2 / (1 - q) * (x - xkp1) ** 2
t = (1 - q) * A / (gam + 2 * q * A)
reg_x = llam_x + gam * L / 2 / (1 - q) / A * (x - z0) ** 2
C0_x, D0_x = C(z0, gy0, fy0, x), D(z0, gy0, fy0, x)                  # first iteration: y_0 = z_0
value = elam - (1 - q) / 4 / m * glam ** 2                          # e_lambda - t/(2L) |g_lambda|^2 for gamma = 0
lam_oqa = (2 * m / (1 - q) * (e1 - e0) - g0 * (g1 - g0)) / (g1 - g0) ** 2     # weight of (5.40), before projection

(check(llam_x - ukp1 - ((1 - lam) * wk_x + lam * C(yk, gyk, fyk, x) - D(yk, gyk, fyk, x))),
 check(llam_x - ukp1 - (elam + glam * (x - z0) + m / (1 - q) * (x - z0) ** 2)),
 check(sp.diff(C(yk, gyk, fyk, x) - D(yk, gyk, fyk, x), x).subs(x, z0) - g1),
 check(sp.diff(reg_x, x).subs(x, z0 - t / L * glam)),
 check(reg_x.subs(x, z0 - t / L * glam) - ukp1 - (elam - t / 2 / L * glam ** 2)),
 check(gam * t / (1 - q - 2 * q * t) - A),
 check(2 * L * (C0_x - D0_x).subs(x, z0) / sp.diff(C0_x - D0_x, x).subs(x, z0) ** 2 - t.subs({gam: 1, A: 2 / (1 - q) ** 2})),
 check(t.subs(gam, 0) - (1 - q) / 2 / q),
 check((z0 - t / L * glam).subs(gam, 0) - zlam),
 check((elam - t / 2 / L * glam ** 2).subs(gam, 0) - value),
 check(sp.diff(value, lam, 2) + (1 - q) / 2 / m * (g1 - g0) ** 2),
 check(sp.diff(value, lam).subs(lam, lam_oqa)),
 check((1 - q) / m * (ukp1 - llam_x.subs(x, zlam)) - R2lam),
 check(R2lam + (1 - q) / m * value))

(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0)

### 4.4 Algorithm 5.11

With $\gamma=0$, the weight above gives the following method, whose display is (5.40).

**Algorithm 5.11. (Triple momentum with quadratic averaging, $m>0$)** Set $x_0=z_0$, and iterate for $k=0,1,\ldots$, with $e_0$, $e_1$, $\bar g_0$ and $\bar g_1$ as in (5.39):
$$
 \left\{\,
 \begin{aligned}
 y_k &= x_k+\tfrac{2\sqrt q}{1+\sqrt q}\,(z_k-x_k),\\
 x_{k+1} &= y_k-\tfrac1L\nabla f(y_k),\\
 \lambda_k &= \mathrm{Proj}_{[0,1]}\bigg\{\frac{\tfrac{2m}{1-q}\,(e_1-e_0)-\langle\bar g_0,\,\bar g_1-\bar g_0\rangle}{\|\bar g_1-\bar g_0\|^2}\bigg\},\\
 w_{k+1} &= (1-\lambda_k)\,w_k+\lambda_k\,C_k,\\
 z_{k+1} &= z_0-\tfrac{1-q}{2m}\,\bar g_{\lambda_k},
 \end{aligned}
 \right.
$$
with the convention $\lambda_0=1$, that is, $w_1=C_0$.

The pair $(w_{k+1}(z_0),\nabla w_{k+1}(z_0))$ follows by the same convex combination, and the minimizer belongs to the ball (5.37) centered at $z_{k+1}$, whose squared radius $R_{k+1}^2=\|z_{k+1}-z_0\|^2-\tfrac{1-q}{m}\,e_{\lambda_k}$ is computed along the way (last two checks of 4.3). For the fixed weight $\lambda=1-(1-\sqrt q)^2$ of (5.38), the first inequality in the proof of Theorem 5.9 gives $R_{k+1}^2\leq(1-\sqrt q)^2R_k^2$ regardless of the previous weights, and the selected weight can only do better. Algorithm 5.11 thus keeps the guarantee of the triple momentum method,
$$
 \|z_N-x_\star\|^2\leq R_N^2\leq(1-\sqrt q)^{2(N-1)}\,\tfrac14\big(\tfrac1m-\tfrac1L\big)^2\|\nabla f(y_0)\|^2 .
$$

In the code, one iteration starts from the state of the previous one, that is, the point $x_k$, the model $l_k=v_k+\tfrac{m}{1-q}\|\cdot-z_k\|^2$ (so that $w_k=l_k-\tfrac{m}{2(1-q)}\|\cdot-x_k\|^2$) and the squared radius $R_k^2=\tfrac{1-q}{m}\big(u_k(x_k)-v_k\big)$, where $z_k$ and $v_k$ are free symbols. The checks are, in this order:

- the first inequality of the proof says that the new lower piece is below the upper bound at $x_k$: it is $C_k(x_k)-D_{k-1}(x_k)\leq0$;
- for the fixed weight $\lambda=1-(1-\sqrt q)^2$, the new center is $(1-\sqrt q)\,z_k+\sqrt q\,\big(y_k-\tfrac1m\nabla f(y_k)\big)$, as in (5.38), and the new squared radius is $(1-\sqrt q)^2\big(R_k^2+\tfrac{1-q}{m}\,(C_k(x_k)-D_{k-1}(x_k))\big)$, hence at most $(1-\sqrt q)^2R_k^2$ whatever $z_k$ and $v_k$ are;
- the two balls behind the weight: for $\lambda=0$, the previous ball adjusted to the new upper bound, with center $p_0=z_k+\tfrac12(x_{k+1}-x_k)$ and squared radius $r_0^2$; for $\lambda=1$, the ball (3.17a) of the new query, with center $p_1=y_k-\tfrac12\big(\tfrac1m+\tfrac1L\big)\nabla f(y_k)$ and squared radius $r_1^2=\tfrac14\big(\tfrac1m-\tfrac1L\big)^2\|\nabla f(y_k)\|^2$, which is the first ball with the convention $\lambda_0=1$;
- for any weight, the center is $(1-\lambda)\,p_0+\lambda\,p_1$ and the squared radius is $(1-\lambda)\,r_0^2+\lambda\,r_1^2-\lambda(1-\lambda)\|p_0-p_1\|^2$: these are the balls $\{x:\ (1-\lambda)(\|x-p_0\|^2-r_0^2)+\lambda(\|x-p_1\|^2-r_1^2)\leq0\}$, which contain the intersection of the two, and the weight of (5.40) selects the smallest of them.

In [22]:
# one iteration of Algorithm 5.11 from the state (x_k, z_k, v_k) of the previous one
yk_tmm = xk + 2 * sq / (1 + sq) * (zk - xk)
xkp1_tmm = yk_tmm - gyk / L
wk_state = lk_x - m / 2 / (1 - q) * (x - xk) ** 2                   # aggregate w_k
_, _, _, _, _, _, z_new, R2_new = ball(yk_tmm, wk_state.subs(x, z0), sp.diff(wk_state, x).subs(x, z0))
R2k = (1 - q) / m * (uk - vk)                                       # squared radius R_k^2
inequality1_tmm = fyk - fykm1 + gyk * (ykm1 - yk_tmm) + (gykm1 - gyk) ** 2 / 2 / L + m / 2 / (1 - q) * (xk - xkp1_tmm) ** 2
lam_tmm = 1 - (1 - sq) ** 2                                         # fixed weight of (5.38)
p0, p1 = z_new.subs(lam, 0), z_new.subs(lam, 1)                     # centers of the two balls
r0, r1 = R2_new.subs(lam, 0), R2_new.subs(lam, 1)                   # and their squared radii

(check(inequality1_tmm - (C(yk_tmm, gyk, fyk, xk) - D(ykm1, gykm1, fykm1, xk))),
 check(z_new.subs(lam, lam_tmm) - ((1 - sq) * zk + sq * (yk_tmm - gyk / m))),
 check(R2_new.subs(lam, lam_tmm) - (1 - sq) ** 2 * (R2k + (1 - q) / m * inequality1_tmm)),
 check(p0 - (zk + (xkp1_tmm - xk) / 2)),
 check(p1 - (yk_tmm - (1 / m + 1 / L) / 2 * gyk)),
 check(r1 - (1 / m - 1 / L) ** 2 * gyk ** 2 / 4),
 check(z_new - ((1 - lam) * p0 + lam * p1)),
 check(R2_new - ((1 - lam) * r0 + lam * r1 - lam * (1 - lam) * (p0 - p1) ** 2)))

(0, 0, 0, 0, 0, 0, 0, 0)